# 02 · Tick-to-trade latency

Runs only the `tick_to_trade` suite: one new bar in, `PortfolioManager::process_market_data()` → `get_recent_executions()` timed. In-process, mock database, mock broker -- this is engine decision latency, not wire latency to a real broker. See docs/superpowers/specs/2026-08-21-tier1-benchmarking-design.md section 3.1.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "python"))
import os; os.chdir(ROOT)
print("repo root:", ROOT)

In [ ]:
from algogauge import manifest, runner, cli
m = manifest.load(ROOT / "algogauge.toml")
suite = m.get("tick_to_trade")
res = runner.run_suite(suite, m.defaults, ROOT, skip_perf=False)
print(f"run {res.run_id}, perf={'on' if res.perf_ran else 'off'}")

## Results by symbol-universe size

In [ ]:
from IPython.display import Markdown, display
display(Markdown(cli.summary_markdown(res.records)))

## Latency vs. universe size

Median and p95 microseconds per tick, one point per `symbols` variant (1 / 8 / 32 / 128), showing how decision latency scales with the size of the traded universe.

In [ ]:
import pandas as pd
import plotly.express as px

rows = [
    {"symbols": int(r.param), "median_us": r.median, "p95_us": r.p95, "executions_per_tick": r.counters.get("executions_per_tick")}
    for r in res.records
    if r.param is not None
]
df = pd.DataFrame(rows).sort_values("symbols")
display(df)
fig = px.line(df.melt(id_vars="symbols", value_vars=["median_us", "p95_us"]),
             x="symbols", y="value", color="variable", markers=True,
             title="Tick-to-trade latency vs. symbol-universe size", log_x=True)
fig.show()

## Flamegraph (if perf ran)

In [ ]:
flamegraph = res.result_dir / "flamegraph.svg"
if flamegraph.exists():
    from IPython.display import SVG, display as _display
    _display(SVG(filename=str(flamegraph)))
else:
    print("no flamegraph for this run (perf was skipped or unavailable)")